In [1]:
# PySpark SQL oturumu için gerekli modülü içe aktarıyoruz
from pyspark.sql import SparkSession

# Veri işleme fonksiyonlarını içe aktarıyoruz
# col       : kolon seçimi ve işlemleri
# when      : koşullu dönüşümler
# count     : satır sayımı
# avg       : ortalama hesaplama
# spark_round: yuvarlama işlemi
# from_json : Kafka'dan gelen JSON mesajlarını parse etme
from pyspark.sql.functions import (
    col, when, count, avg,
    round as spark_round,
    from_json
)

# Veri tipi tanımlamaları
# StructType/StructField : şema tanımı için
# StringType  : metin kolonlar
# IntegerType : tam sayı kolonlar
# FloatType   : ondalıklı sayı kolonlar
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    FloatType
)

# Sistem işlemleri ve bekleme için standart kütüphaneler
import os
import time

print("✅ Kütüphaneler başarıyla yüklendi!")

✅ Kütüphaneler başarıyla yüklendi!


In [2]:
# Docker ortamında Spark oturumunu başlatıyoruz
# configure_spark_with_delta_pip kullanmıyoruz
# çünkü Delta Lake Docker container'ında zaten kurulu
spark = (
    SparkSession.builder
    .appName("Spotify_DeltaLake_Pipeline")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark oturumu başarıyla başlatıldı!")
print(f"Spark versiyonu: {spark.version}")

✅ Spark oturumu başarıyla başlatıldı!
Spark versiyonu: 3.5.0


In [3]:
# Docker container içindeki klasör yollarını tanımlıyoruz
# docker-compose.yml'deki volume mount'lara göre ayarlandı:
# ./spark/notebooks → /home/jovyan/notebooks
# delta-lake-data   → /home/jovyan/delta-lake

import os

# Bronze katmanı: ham veri
BRONZE_YOL = "/home/jovyan/delta-lake/bronze"
# Silver katmanı: temizlenmiş veri
SILVER_YOL = "/home/jovyan/delta-lake/silver"
# Gold katmanı: analiz için hazır özet veri
GOLD_YOL   = "/home/jovyan/delta-lake/gold"
# Checkpoint: Spark streaming ilerleme kaydı
CHECKPOINT_YOL = "/home/jovyan/delta-lake/checkpoints"

print("✅ Klasör yolları tanımlandı!")
print(f"🥉 Bronze     : {BRONZE_YOL}")
print(f"🥈 Silver     : {SILVER_YOL}")
print(f"🥇 Gold       : {GOLD_YOL}")
print(f"📁 Checkpoint : {CHECKPOINT_YOL}")

✅ Klasör yolları tanımlandı!
🥉 Bronze     : /home/jovyan/delta-lake/bronze
🥈 Silver     : /home/jovyan/delta-lake/silver
🥇 Gold       : /home/jovyan/delta-lake/gold
📁 Checkpoint : /home/jovyan/delta-lake/checkpoints


In [4]:
# Kafka Producer'dan gelen mesajların şemasını tanımlıyoruz
# Producer'ın gönderdiği tüm alanlar burada tanımlanmıştır
# Kaynak: kafka/producer/producer.py
SPOTIFY_SCHEMA = StructType([
    # Kafka simülasyon alanları
    StructField("kafka_timestamp",   StringType(),  True),  # Mesaj gönderim zamanı
    StructField("user_id",           StringType(),  True),  # Rastgele üretilen kullanıcı ID
    StructField("event_type",        StringType(),  True),  # Olay tipi: "track_played"

    # Şarkı kimlik bilgileri
    StructField("track_id",          StringType(),  True),  # Spotify şarkı ID
    StructField("track_name",        StringType(),  True),  # Şarkı adı
    StructField("artists",           StringType(),  True),  # Sanatçı adları
    StructField("album_name",        StringType(),  True),  # Albüm adı
    StructField("track_genre",       StringType(),  True),  # Müzik türü (ML hedefi)

    # Sayısal özellikler
    StructField("popularity",        IntegerType(), True),  # Popülerlik (0-100)
    StructField("duration_ms",       IntegerType(), True),  # Süre (milisaniye)
    StructField("explicit",          StringType(),  True),  # Explicit içerik (True/False)
    StructField("danceability",      FloatType(),   True),  # Dans edilebilirlik (0-1)
    StructField("energy",            FloatType(),   True),  # Enerji (0-1)
    StructField("key",               IntegerType(), True),  # Müzik tonu
    StructField("loudness",          FloatType(),   True),  # Ses yüksekliği (dB)
    StructField("mode",              IntegerType(), True),  # Major/Minor (1/0)
    StructField("speechiness",       FloatType(),   True),  # Konuşma oranı (0-1)
    StructField("acousticness",      FloatType(),   True),  # Akustiklik (0-1)
    StructField("instrumentalness",  FloatType(),   True),  # Enstrümantal oran (0-1)
    StructField("liveness",          FloatType(),   True),  # Canlı performans (0-1)
    StructField("valence",           FloatType(),   True),  # Duygusal ton (0-1)
    StructField("tempo",             FloatType(),   True),  # Tempo (BPM)
    StructField("time_signature",    IntegerType(), True),  # Zaman imzası
])

print("✅ Şema tanımlandı!")
print(f"Toplam kolon sayısı: {len(SPOTIFY_SCHEMA.fields)}")

✅ Şema tanımlandı!
Toplam kolon sayısı: 23


In [5]:
# Kafka'dan Spark Structured Streaming ile veri okuyoruz
# Docker network içinde kafka:29092 adresi kullanılıyor
# (docker-compose.yml'deki PLAINTEXT://kafka:29092 listener)
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "spotify-tracks")
    # En baştan okumaya başla (tüm mesajları al)
    .option("startingOffsets", "earliest")
    .load()
)

# Kafka'dan gelen binary value'yu string'e çeviriyoruz
# Sonra SPOTIFY_SCHEMA'ya göre JSON parse ediyoruz
raw_df = (
    raw_df
    .select(
        from_json(
            raw_df.value.cast("string"),
            SPOTIFY_SCHEMA
        ).alias("data")
    )
    .select("data.*")
)

print("✅ Kafka stream bağlantısı tanımlandı!")
print(f"Topic  : spotify-tracks")
print(f"Broker : kafka:29092")

✅ Kafka stream bağlantısı tanımlandı!
Topic  : spotify-tracks
Broker : kafka:29092


In [6]:
# BRONZE KATMANI: Ham veriyi Delta Lake'e yazıyoruz
# Kafka'dan gelen veri hiç işlenmeden Bronze'a yazılır
# writeStream: sürekli akan veriyi Delta Lake'e yazar
# outputMode("append"): yeni gelen verileri ekler, silmez
# checkpointLocation: hata durumunda kaldığı yerden devam eder
# trigger(processingTime): her 30 saniyede bir batch işle
# → Delta Lake'e büyük dosyalar yazar, small files problemi olmaz

bronze_query = (
    raw_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{CHECKPOINT_YOL}/bronze"
    )
    .trigger(processingTime="30 seconds")
    .start(BRONZE_YOL)
)

# Producer tüm veriyi gönderene kadar bekliyoruz
# timeout=300 → 5 dakika sonra otomatik durur
# 500 msg/sn × 300 sn = 150,000 mesaj → 114K için yeterli
print("⏳ 5 dakika boyunca Kafka'dan veri okunuyor...")
print("⚠️ Producer hızı: 500 msg/sn → ~4 dakikada 114K mesaj gelir")
bronze_query.awaitTermination(timeout=300)

# Streaming'i durduruyoruz
bronze_query.stop()

# Doğrulama: Bronze'dan okuyup satır sayısını kontrol ediyoruz
bronze_kontrol = spark.read.format("delta").load(BRONZE_YOL)
print(f"✅ Bronze katmanına yazıldı!")
print(f"📁 Konum          : {BRONZE_YOL}")
print(f"📊 Satır sayısı   : {bronze_kontrol.count():,}")

⏳ Bronze dolana kadar 30 saniye bekleniyor...
✅ Bronze katmanına yazıldı!
📁 Konum          : /home/jovyan/delta-lake/bronze
📊 Satır sayısı   : 5,015


In [ ]:
# SILVER KATMANI İÇİN VERİ TEMİZLEME
# Bronze'dan okunan ham veri üzerinde 4 aşamalı temizleme yapıyoruz

print("=== TEMİZLEME ÖNCESİ NULL SAYILARI ===")
bronze_kontrol.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_kontrol.columns
]).show()

# Aşama 1: Kolonlarda null değer varsa o satırı atıyoruz
temiz_veri = bronze_kontrol.dropna()
print(f"Aşama 1 - Null temizleme    : {temiz_veri.count():,} satır kaldı")

# Aşama 2: Aynı track_id'ye sahip duplike kayıtları kaldırıyoruz
temiz_veri = temiz_veri.dropDuplicates(["track_id"])
print(f"Aşama 2 - Duplike temizleme : {temiz_veri.count():,} satır kaldı")

# Aşama 3: Popularity değeri 0-100 aralığı dışındakileri filtreliyoruz
temiz_veri = temiz_veri.filter(
    (col("popularity") >= 0) & (col("popularity") <= 100)
)
print(f"Aşama 3 - Popularity filtre : {temiz_veri.count():,} satır kaldı")

# Aşama 4: explicit kolonunu String'den Integer'a çeviriyoruz
# "True" → 1, "False" → 0 (ML modeli sayısal değer ister)
temiz_veri = temiz_veri.withColumn(
    "explicit",
    when(col("explicit") == "True", 1)
    .when(col("explicit") == "False", 0)
    .otherwise(0)
)

print("\n✅ Veri temizleme tamamlandı!")

=== TEMİZLEME ÖNCESİ NULL SAYILARI ===
+---------------+-------+----------+--------+----------+-------+----------+-----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+
|kafka_timestamp|user_id|event_type|track_id|track_name|artists|album_name|track_genre|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|
+---------------+-------+----------+--------+----------+-------+----------+-----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+
|              0|      0|         0|       0|         0|      0|         0|          0|         0|          0|       0|           0|     0|  0|       0|   0|          0|           0|               0|       0|      0|    0|             0|
+--------

In [8]:
# Temizlenmiş veriyi Silver katmanına yazıyoruz
# Batch yazma kullanıyoruz çünkü Bronze'dan zaten okuduk
(
    temiz_veri.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_YOL)
)

# Doğrulama: Silver'dan okuyup kontrol ediyoruz
silver_kontrol = spark.read.format("delta").load(SILVER_YOL)
print(f"✅ Silver katmanına yazıldı!")
print(f"📁 Konum        : {SILVER_YOL}")
print(f"📊 Satır sayısı : {silver_kontrol.count():,}")
silver_kontrol.show(3)

✅ Silver katmanına yazıldı!
📁 Konum        : /home/jovyan/delta-lake/silver
📊 Satır sayısı : 4,419
+--------------------+--------------------+------------+--------------------+--------------+--------------------+--------------------+-----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+
|     kafka_timestamp|             user_id|  event_type|            track_id|    track_name|             artists|          album_name|track_genre|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|
+--------------------+--------------------+------------+--------------------+--------------+--------------------+--------------------+-----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+
|202

In [9]:
# GOLD KATMANI: Analiz için hazır özet veri oluşturuyoruz
# Tür (track_genre) bazında müzik istatistikleri hesaplanıyor
# Bu tablo Kişi 3 (EDA) ve Kişi 4 (ML) tarafından kullanılacak

genre_summary = (
    silver_kontrol
    .groupBy("track_genre")
    .agg(
        count("track_id").alias("sarki_sayisi"),
        spark_round(avg("popularity"), 2).alias("ort_popularity"),
        spark_round(avg("danceability"), 4).alias("ort_danceability"),
        spark_round(avg("energy"), 4).alias("ort_energy"),
        spark_round(avg("tempo"), 2).alias("ort_tempo"),
        spark_round(avg("loudness"), 2).alias("ort_loudness"),
        spark_round(avg("valence"), 4).alias("ort_valence")
    )
    .orderBy(col("ort_popularity").desc())
)

# Gold katmanına yazıyoruz
(
    genre_summary.write
    .format("delta")
    .mode("overwrite")
    .save(GOLD_YOL)
)

# Doğrulama
gold_kontrol = spark.read.format("delta").load(GOLD_YOL)
print(f"✅ Gold katmanına yazıldı!")
print(f"📁 Konum        : {GOLD_YOL}")
print(f"🎵 Toplam tür   : {gold_kontrol.count()}")
print("\nEn popüler 10 tür:")
gold_kontrol.show(10)

✅ Gold katmanına yazıldı!
📁 Konum        : /home/jovyan/delta-lake/gold
🎵 Toplam tür   : 6

En popüler 10 tür:
+-----------+------------+--------------+----------------+----------+---------+------------+-----------+
|track_genre|sarki_sayisi|ort_popularity|ort_danceability|ort_energy|ort_tempo|ort_loudness|ort_valence|
+-----------+------------+--------------+----------------+----------+---------+------------+-----------+
|      anime|          15|         69.07|          0.5258|    0.8331|   115.65|       -4.42|     0.4803|
|    ambient|         999|         44.21|           0.368|    0.2373|   111.16|       -18.6|     0.1673|
|   acoustic|        1000|         42.48|          0.5496|    0.4354|   119.01|       -9.45|      0.424|
|   alt-rock|         999|          33.9|          0.5346|     0.754|   124.65|       -6.19|     0.5182|
|   afrobeat|         999|         24.41|          0.6694|    0.7029|   119.24|       -7.79|     0.6985|
|alternative|         407|         22.22|        

In [10]:
# Tüm Delta Lake katmanlarının özet raporunu yazdırıyoruz
print("=" * 55)
print("       DELTA LAKE PIPELINE ÖZET RAPORU")
print("=" * 55)

bronze_df = spark.read.format("delta").load(BRONZE_YOL)
silver_df = spark.read.format("delta").load(SILVER_YOL)
gold_df   = spark.read.format("delta").load(GOLD_YOL)

print(f"🥉 Bronze  : {bronze_df.count():>8,} satır  (ham veri)")
print(f"🥈 Silver  : {silver_df.count():>8,} satır  (temizlenmiş)")
print(f"🥇 Gold    : {gold_df.count():>8,} satır  (tür özeti)")
print(f"\n🗑️  Temizlenen : {bronze_df.count() - silver_df.count():,} kirli kayıt")
print(f"🎵 Toplam tür : {gold_df.count()} farklı müzik türü")
print("=" * 55)
print("✅ Adım 3 tamamlandı! Delta Lake pipeline hazır.")
print("   Kişi 3 EDA için Silver katmanını kullanabilir.")
print("   Kişi 4 ML için Gold katmanını kullanabilir.")

       DELTA LAKE PIPELINE ÖZET RAPORU
🥉 Bronze  :    5,015 satır  (ham veri)
🥈 Silver  :    4,419 satır  (temizlenmiş)
🥇 Gold    :        6 satır  (tür özeti)

🗑️  Temizlenen : 596 kirli kayıt
🎵 Toplam tür : 6 farklı müzik türü
✅ Adım 3 tamamlandı! Delta Lake pipeline hazır.
   Kişi 3 EDA için Silver katmanını kullanabilir.
   Kişi 4 ML için Gold katmanını kullanabilir.


In [11]:
# ÖNEMLİ NOT: Kafka Entegrasyonu Hakkında
# Bu notebook Docker ortamında çalışmak üzere tasarlanmıştır.
# Kafka Producer (kafka/producer/producer.py) çalışırken
# bu notebook spotify-tracks topic'ini dinler.
#
# Kafka bağlantı bilgileri:
# Bootstrap Server : kafka:29092 (Docker iç ağı)
# Topic            : spotify-tracks
# Offset           : earliest (en baştan okur)
#
# Docker olmadan test için localhost:9092 kullanılabilir.

print("✅ Adım 3 notebook'u tamamlandı!")
print("Kafka broker : kafka:29092")
print("Topic        : spotify-tracks")

✅ Adım 3 notebook'u tamamlandı!
Kafka broker : kafka:29092
Topic        : spotify-tracks
